In [1]:
import os
from glob import glob
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, random_split

# ---------------------------------------------------------
# 1. 커스텀 데이터셋 (LeapGestRecog 맞춤형 - 변경 없음)
# ---------------------------------------------------------
class LeapGestureDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        self.classes = sorted([
            '01_palm', '02_l', '03_fist', '04_fist_moved', '05_thumb',
            '06_index', '07_ok', '08_palm_moved', '09_c', '10_down'
        ])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        for img_path in glob(os.path.join(root_dir, '*', '*', '*.png')):
            gesture_name = os.path.basename(os.path.dirname(img_path))
            if gesture_name in self.class_to_idx:
                self.image_paths.append(img_path)
                self.labels.append(self.class_to_idx[gesture_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB') # 흑백 -> 3채널 자동 복제
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# ---------------------------------------------------------
# 2. 하드웨어 가속기 및 하이퍼파라미터 설정
# ---------------------------------------------------------
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"현재 구동 중인 디바이스: {device}")

BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ---------------------------------------------------------
# 3. 데이터 로드 및 분할
# ---------------------------------------------------------
data_dir = './leapGestRecog' # ★ 데이터 폴더 경로 확인

print("데이터를 불러오는 중...")
full_dataset = LeapGestureDataset(root_dir=data_dir, transform=transform)
print(f"총 이미지 개수: {len(full_dataset)}장")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ---------------------------------------------------------
# 4. 모델 설정 (EfficientNet-B0 바닥부터 학습) ★ 핵심 변경 포인트
# ---------------------------------------------------------
print("EfficientNet-B0 모델을 준비합니다...")
model = models.efficientnet_b0(weights=None) # 파인튜닝 아님, 백지 상태

# EfficientNet은 마지막 분류기가 'classifier'라는 이름의 Sequential로 묶여 있습니다.
# classifier[0]은 Dropout, classifier[1]이 Linear(선형) 층입니다.
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 10) # 10개 클래스로 변경

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ---------------------------------------------------------
# 5. 본격적인 학습 및 검증 루프
# ---------------------------------------------------------
print("\n--- 학습 시작 ---")
for epoch in range(1, EPOCHS + 1):
    
    # [Train Phase]
    model.train()
    running_loss, train_correct, train_total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        
    train_loss = running_loss / len(train_loader)
    train_acc = 100. * train_correct / train_total
    
    # [Validation Phase]
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_acc = 100. * val_correct / val_total
    
    print(f"Epoch [{epoch:2d}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("--- 학습 완료 ---")

현재 구동 중인 디바이스: mps
데이터를 불러오는 중...
총 이미지 개수: 20000장
EfficientNet-B0 모델을 준비합니다...

--- 학습 시작 ---
Epoch [ 1/10] | Train Loss: 0.3437 | Train Acc: 88.61% | Val Acc: 99.17%
Epoch [ 2/10] | Train Loss: 0.0487 | Train Acc: 98.51% | Val Acc: 100.00%
Epoch [ 3/10] | Train Loss: 0.0185 | Train Acc: 99.46% | Val Acc: 99.88%
Epoch [ 4/10] | Train Loss: 0.0234 | Train Acc: 99.39% | Val Acc: 98.25%
Epoch [ 5/10] | Train Loss: 0.0241 | Train Acc: 99.28% | Val Acc: 99.80%
Epoch [ 6/10] | Train Loss: 0.0012 | Train Acc: 99.98% | Val Acc: 100.00%
Epoch [ 7/10] | Train Loss: 0.0170 | Train Acc: 99.51% | Val Acc: 99.90%
Epoch [ 8/10] | Train Loss: 0.0220 | Train Acc: 99.41% | Val Acc: 100.00%
Epoch [ 9/10] | Train Loss: 0.0068 | Train Acc: 99.81% | Val Acc: 99.97%
Epoch [10/10] | Train Loss: 0.0193 | Train Acc: 99.53% | Val Acc: 86.17%
--- 학습 완료 ---
